# BizLens v2.2.12 - Logistic Regression (Comprehensive)

Predict survival on the Titanic using **BizLens + Logistic Regression**.
Covers data exploration, model building, interpretation, and evaluation.

In [ ]:
import subprocess
import sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "bizlens"])
print("✅ BizLens v2.2.14 ready for Logistic Regression!")

In [ ]:
import bizlens as bl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score, roc_curve

sns.set_style("whitegrid")
# bl.set_profiling(True)   # Optional performance timing

# ── Optional: Use a BizLens built-in dataset ─────────────────────────────
# df = bl.load_dataset('titanic')  # predict survival (survived) from age, pclass, sex, fare
# df = bl.load_dataset('penguins')  # predict species from bill and body measurements
#
# After loading, explore with:
# print(df.shape)         # rows × columns
# print(df.dtypes)        # column types
# print(df.head())        # first 5 rows
#
# ── Export / Save Results ─────────────────────────────────────────────────
# pd.DataFrame({'actual': y_test, 'predicted': y_pred}).to_csv('classification_results.csv', index=False)
# df.to_excel('output.xlsx', index=False)  # Save as Excel
# df.to_json('output.json', orient='records', indent=2)  # Save as JSON
# ── Polars alternative (optional — for users who prefer polars) ──────────────
# import polars as pl
# df_pl = pl.from_pandas(bl.load_dataset('titanic'))    # load → convert to polars
# df_pl = pl.read_csv('your_data.csv')                  # or read directly
# df_pl = pl.DataFrame(data)                             # or create from dict
#
# BizLens accepts both pandas and polars DataFrames transparently:
# bl.tables.frequency_table(df_pl['sex'])               # works with polars too
# bl.quality.data_profile(df_pl)                        # works with polars too
#
# Convert polars back to pandas when needed:
# df = df_pl.to_pandas()

## 1. Load Titanic Dataset using BizLens

In [ ]:
df = bl.load_dataset("titanic")
print(f"Dataset shape: {df.shape}")
bl.describe(df)

## 2. Data Quality & Exploration with BizLens

In [ ]:
bl.quality.data_profile(df)
bl.quality.completeness_report(df)

In [ ]:
# Survival rate by key variables
bl.tables.contingency_table(df, 'sex', 'survived')[0]
bl.tables.contingency_table(df, 'pclass', 'survived')[0]

## 3. Data Preprocessing

In [ ]:
# Select features and target
df_model = df[['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare']].copy()

# Handle missing values
df_model['age'] = df_model['age'].fillna(df_model['age'].median())

# Encode categorical variables
df_model = pd.get_dummies(df_model, columns=['sex', 'pclass'], drop_first=True)

X = df_model.drop('survived', axis=1)
y = df_model['survived']

print("Features prepared for modeling.")

## 4. Logistic Regression Model

In [ ]:
X_const = sm.add_constant(X)
logit_model = sm.Logit(y, X_const).fit()
print(logit_model.summary())

In [ ]:
# Odds Ratios
odds_ratios = np.exp(logit_model.params)
print("\n=== Odds Ratios ===")
print(odds_ratios.sort_values(ascending=False))

## 5. Model Evaluation

In [ ]:
y_pred_prob = logit_model.predict(X_const)
y_pred = (y_pred_prob >= 0.5).astype(int)

print("Classification Report:")
print(classification_report(y, y_pred))

print(f"ROC-AUC Score: {roc_auc_score(y, y_pred_prob):.4f}")

## 6. Visualizations

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

In [ ]:
# ROC Curve
fpr, tpr, _ = roc_curve(y, y_pred_prob)
plt.plot(fpr, tpr, label=f'AUC = {roc_auc_score(y, y_pred_prob):.3f}')
plt.plot([0,1], [0,1], 'r--')
plt.title('ROC Curve')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend()
plt.show()